# FASTTEXT SENTIMENT ANALYSIS - AUTO OPTIMIZATION
Notebook untuk training model sentiment analysis menggunakan FastText dengan automatic hyperparameter optimization.

# 1. IMPORT LIBRARY

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import random
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

import fasttext

print("✓ Library berhasil diimport")

ModuleNotFoundError: No module named 'seaborn'

# 2. LOAD DATA

In [ ]:
# Load dataset
csv_file = os.path.join('..', 'data', 'Data Label', 'LPDP_cleaned_label.csv')
data_label = pd.read_csv(csv_file)

print(f"Dataset loaded: {len(data_label)} rows")
print(f"Columns: {list(data_label.columns)}")
print(f"\nFirst 5 rows:")
data_label.head()

# 3. PREPROCESSING

In [ ]:
# Pilih kolom yang dibutuhkan
data_label = data_label[['normalisasi_text', 'label_voting']]

# Hapus data yang kosong
print(f"Data sebelum cleaning: {len(data_label)}")
data_label = data_label.dropna()
print(f"Data setelah cleaning: {len(data_label)}")

# Lihat distribusi label
print("\n" + "="*60)
print("DISTRIBUSI LABEL:")
print("="*60)
print(data_label['label_voting'].value_counts())
print(f"\nTotal data: {len(data_label)}")

data_label.head()

# 4. SPLIT DATA

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    data_label['normalisasi_text'], 
    data_label['label_voting'],
    test_size=0.2,
    random_state=42,
    stratify=data_label['label_voting']
)

print(f"Train size: {len(X_train)} ({len(X_train)/len(data_label)*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/len(data_label)*100:.1f}%)")

print("\nDistribusi label di train set:")
print(y_train.value_counts())
print("\nDistribusi label di test set:")
print(y_test.value_counts())

# 5. CONVERT KE FORMAT FASTTEXT

In [ ]:
# Function untuk save data ke format FastText
def save_to_fasttext_format(texts, labels, filename):
    """
    Save data ke format FastText: __label__<label> <text>
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for text, label in zip(texts, labels):
            # FastText format: __label__<label> <text>
            f.write(f"__label__{label} {text}\n")
    print(f"✓ Data berhasil disimpan ke {filename}")

# Save train dan test data
train_file = 'train_fasttext.txt'
test_file = 'test_fasttext.txt'

save_to_fasttext_format(X_train, y_train, train_file)
save_to_fasttext_format(X_test, y_test, test_file)

# Preview format
print(f"\nPreview {train_file} (3 baris pertama):")
with open(train_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

# 6. SETUP HYPERPARAMETER GRID

In [ ]:
# Hyperparameter combinations untuk optimasi
param_grid = {
    'lr': [0.1, 0.5, 1.0],           # learning rate
    'epoch': [25, 50, 100],          # jumlah epoch
    'wordNgrams': [1, 2, 3],         # n-grams (1=unigram, 2=bigram, 3=trigram)
    'dim': [50, 100, 200],           # dimensi embedding
    'ws': [3, 5, 7],                 # window size
    'minCount': [1, 2, 5]            # minimum word count
}

print("PARAMETER GRID UNTUK OPTIMIZATION:")
print("="*60)
for param, values in param_grid.items():
    print(f"{param:15s}: {values}")
print("="*60)

# Hitung total kemungkinan kombinasi
total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nTotal kemungkinan kombinasi: {total_combinations}")

# 7. RANDOM SEARCH OPTIMIZATION

In [ ]:
# Generate random combinations (lebih cepat dari GridSearch)
n_combinations = 30  # jumlah kombinasi yang akan dicoba

print(f"Akan mencoba {n_combinations} dari {total_combinations} kemungkinan kombinasi")
print("Menggunakan RandomizedSearch untuk efisiensi...\n")

# Random sampling dari parameter grid
random_params = []
random.seed(42)  # untuk reproducibility

for i in range(n_combinations):
    params = {
        'lr': random.choice(param_grid['lr']),
        'epoch': random.choice(param_grid['epoch']),
        'wordNgrams': random.choice(param_grid['wordNgrams']),
        'dim': random.choice(param_grid['dim']),
        'ws': random.choice(param_grid['ws']),
        'minCount': random.choice(param_grid['minCount'])
    }
    random_params.append(params)

print(f"✓ {n_combinations} kombinasi parameter telah digenerate")

# 8. TRAINING & OPTIMIZATION
⚠️ **Cell ini akan memakan waktu cukup lama**. Progress akan ditampilkan untuk setiap kombinasi yang dicoba.

In [ ]:
# Training dengan berbagai kombinasi parameter
best_accuracy = 0
best_params = None
best_model = None
results = []

print("MULAI TRAINING & OPTIMIZATION")
print("="*60)

for idx, params in enumerate(random_params, 1):
    print(f"\n{'='*60}")
    print(f"Kombinasi {idx}/{n_combinations}")
    print(f"Params: lr={params['lr']}, epoch={params['epoch']}, wordNgrams={params['wordNgrams']}")
    print(f"        dim={params['dim']}, ws={params['ws']}, minCount={params['minCount']}")
    print(f"{'='*60}")
    
    try:
        # Training model
        model = fasttext.train_supervised(
            input=train_file,
            lr=params['lr'],
            epoch=params['epoch'],
            wordNgrams=params['wordNgrams'],
            dim=params['dim'],
            ws=params['ws'],
            minCount=params['minCount'],
            verbose=0  # suppress training output
        )
        
        # Evaluasi pada test set
        n_samples, accuracy = model.test(test_file)
        
        results.append({
            'combination': idx,
            'params': params,
            'accuracy': accuracy,
            'n_samples': n_samples
        })
        
        print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        # Simpan model terbaik
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = params.copy()
            best_model = model
            print(f"⭐ NEW BEST MODEL! Accuracy: {accuracy:.4f}")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        continue

print(f"\n{'='*60}")
print("OPTIMIZATION SELESAI!")
print(f"{'='*60}")
print(f"\n🏆 Best Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"📊 Best Parameters:")
for param, value in best_params.items():
    print(f"   {param:15s}: {value}")

# 9. VISUALISASI HASIL OPTIMIZATION

In [ ]:
# Sort results by accuracy
sorted_results = sorted(results, key=lambda x: x['accuracy'], reverse=True)

# Create DataFrame untuk analisis
results_df = pd.DataFrame([
    {
        'Combination': r['combination'],
        'Accuracy': r['accuracy'],
        'lr': r['params']['lr'],
        'epoch': r['params']['epoch'],
        'wordNgrams': r['params']['wordNgrams'],
        'dim': r['params']['dim'],
        'ws': r['params']['ws'],
        'minCount': r['params']['minCount']
    }
    for r in results
])

print("TOP 10 CONFIGURATIONS:")
print("="*80)
print(results_df.nlargest(10, 'Accuracy').to_string(index=False))

In [ ]:
# Visualisasi 1: Bar chart top 10 models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Top 10 accuracies
ax1 = axes[0, 0]
top_10 = sorted_results[:10]
accuracies = [r['accuracy'] for r in top_10]
combinations = [f"Config {r['combination']}" for r in top_10]

ax1.barh(combinations[::-1], accuracies[::-1], color='skyblue')
ax1.axvline(x=best_accuracy, color='r', linestyle='--', linewidth=2, label=f'Best: {best_accuracy:.4f}')
ax1.set_xlabel('Accuracy', fontsize=12)
ax1.set_ylabel('Configuration', fontsize=12)
ax1.set_title('Top 10 Model Configurations', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Plot 2: Learning rate vs Accuracy
ax2 = axes[0, 1]
for lr in param_grid['lr']:
    lr_results = results_df[results_df['lr'] == lr]
    ax2.scatter([lr]*len(lr_results), lr_results['Accuracy'], alpha=0.6, s=100, label=f'lr={lr}')
ax2.set_xlabel('Learning Rate', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Learning Rate vs Accuracy', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Epoch vs Accuracy
ax3 = axes[1, 0]
for epoch in param_grid['epoch']:
    epoch_results = results_df[results_df['epoch'] == epoch]
    ax3.scatter([epoch]*len(epoch_results), epoch_results['Accuracy'], alpha=0.6, s=100, label=f'epoch={epoch}')
ax3.set_xlabel('Epochs', fontsize=12)
ax3.set_ylabel('Accuracy', fontsize=12)
ax3.set_title('Epochs vs Accuracy', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Plot 4: Embedding Dimension vs Accuracy
ax4 = axes[1, 1]
for dim in param_grid['dim']:
    dim_results = results_df[results_df['dim'] == dim]
    ax4.scatter([dim]*len(dim_results), dim_results['Accuracy'], alpha=0.6, s=100, label=f'dim={dim}')
ax4.set_xlabel('Embedding Dimension', fontsize=12)
ax4.set_ylabel('Accuracy', fontsize=12)
ax4.set_title('Embedding Dimension vs Accuracy', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualisasi berhasil dibuat")

# 10. EVALUASI MODEL TERBAIK

In [ ]:
# Prediksi dengan model terbaik pada test set
print("EVALUASI MODEL TERBAIK PADA TEST SET")
print("="*60)

y_pred = []
y_pred_proba = []

for text in X_test:
    pred = best_model.predict(text)
    # Remove '__label__' prefix
    label = pred[0][0].replace('__label__', '')
    confidence = pred[1][0]
    y_pred.append(label)
    y_pred_proba.append(confidence)

# Convert to numpy arrays
y_test_np = np.array(y_test)
y_pred_np = np.array(y_pred)

print("\n📊 CLASSIFICATION REPORT:")
print("="*60)
print(classification_report(y_test_np, y_pred_np, digits=4))

# Accuracy
acc = accuracy_score(y_test_np, y_pred_np)
print(f"\n🎯 Final Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_np, y_pred_np)
labels = sorted(data_label['label_voting'].unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Best Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 CONFUSION MATRIX:")
print("="*60)
print(cm)

# 11. SAVE MODEL & PARAMETERS

In [ ]:
# Save model terbaik
model_path = 'best_fasttext_model.bin'
best_model.save_model(model_path)
print(f"✓ Model terbaik berhasil disimpan di: {model_path}")

# Save best parameters
params_path = 'best_params.json'
with open(params_path, 'w') as f:
    json.dump(best_params, f, indent=4)
print(f"✓ Best parameters disimpan di: {params_path}")

# Save all results
results_path = 'optimization_results.json'
# Convert results to serializable format
results_serializable = [
    {
        'combination': r['combination'],
        'accuracy': float(r['accuracy']),
        'params': r['params']
    }
    for r in results
]
with open(results_path, 'w') as f:
    json.dump(results_serializable, f, indent=4)
print(f"✓ All optimization results disimpan di: {results_path}")

# Save results as CSV
results_csv_path = 'optimization_results.csv'
results_df.to_csv(results_csv_path, index=False)
print(f"✓ Results CSV disimpan di: {results_csv_path}")

print(f"\n{'='*60}")
print("SEMUA FILE BERHASIL DISIMPAN!")
print(f"{'='*60}")

# 12. LOAD MODEL (Optional)
Gunakan cell ini untuk load model yang sudah disimpan di lain waktu.

In [ ]:
# Load saved model
# model_loaded = fasttext.load_model('best_fasttext_model.bin')
# print("✓ Model berhasil di-load")

# Load parameters
# with open('best_params.json', 'r') as f:
#     params_loaded = json.load(f)
# print("✓ Parameters berhasil di-load")
# print(params_loaded)

# 13. TEST PREDIKSI
Test model dengan contoh kalimat baru.

In [ ]:
# Test dengan contoh kalimat
test_texts = [
    "program lpdp sangat membantu saya kuliah di luar negeri terima kasih",
    "kecewa sekali dengan proses seleksi lpdp yang tidak transparan dan berbelit",
    "informasi pendaftaran lpdp akan dibuka bulan depan",
    "alhamdulillah lolos beasiswa lpdp tahun ini",
    "kapan pengumuman hasil seleksi lpdp",
    "mahal sekali biaya pendaftaran lpdp tidak adil"
]

print("PREDIKSI CONTOH KALIMAT:")
print("="*60)
for i, text in enumerate(test_texts, 1):
    pred = best_model.predict(text)
    label = pred[0][0].replace('__label__', '')
    confidence = pred[1][0]
    
    print(f"\n{i}. Teks: {text}")
    print(f"   Prediksi: {label}")
    print(f"   Confidence: {confidence:.4f} ({confidence*100:.2f}%)")

# 14. INTERACTIVE PREDICTION
Gunakan cell ini untuk prediksi interaktif.

In [ ]:
def predict_sentiment(text):
    """
    Fungsi untuk prediksi sentiment dari text input
    """
    pred = best_model.predict(text)
    label = pred[0][0].replace('__label__', '')
    confidence = pred[1][0]
    
    print(f"Teks: {text}")
    print(f"Prediksi: {label}")
    print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
    return label, confidence

# Contoh penggunaan:
# predict_sentiment("Saya sangat senang dengan program LPDP ini")

# SUMMARY

## Model Performance
- **Best Accuracy**: Lihat hasil di section 8
- **Model Type**: FastText Supervised Learning
- **Optimization Method**: RandomizedSearch
- **Number of Trials**: 30 kombinasi parameter

## Files Generated
1. `best_fasttext_model.bin` - Model terbaik
2. `best_params.json` - Best hyperparameters
3. `optimization_results.json` - Semua hasil optimization
4. `optimization_results.csv` - Hasil optimization dalam format CSV
5. `train_fasttext.txt` - Training data dalam format FastText
6. `test_fasttext.txt` - Test data dalam format FastText

## Next Steps
- Coba dengan lebih banyak kombinasi parameter untuk hasil lebih optimal
- Eksperimen dengan data augmentation
- Coba ensemble dengan model lain (SVM, LSTM, BERT, dll)
- Deploy model untuk production